# Knoema API Server

This notebook demonstrates the Phase 44 REST surface and shows the matching WebSocket endpoint for live log streaming.


In [ ]:
import json
import threading
import time

import httpx
import uvicorn


In [ ]:
config = uvicorn.Config('knoema.api.server:app', host='127.0.0.1', port=8000, log_level='warning')
server = uvicorn.Server(config)
thread = threading.Thread(target=server.run, daemon=True)
thread.start()
time.sleep(1.0)


In [ ]:
payload = {
    'runtime': {
        'duration_days': 1,
        'tick_duration_minutes': 120,
        'prompt_language': 'en',
        'stream_delay_seconds': 0.01,
    },
    'environment': {
        'start_time': '2026-04-19T09:00:00',
        'location_path': ['Korea', 'Seoul', 'Campus', 'Commons'],
        'conditions': {'weather': 'clear'},
    },
    'agents': [
        {
            'agent_id': 'agent-0',
            'name': 'Agent 0',
            'age': 20,
            'background': 'Synthetic participant.',
            'personality': {
                'openness': 0.5,
                'conscientiousness': 0.6,
                'extraversion': 0.4,
                'agreeableness': 0.7,
                'neuroticism': 0.3,
            },
            'values': ['clarity'],
            'goals': ['cooperate'],
            'theory_of_mind': {'enabled': True},
        }
    ],
    'local_response': '{"action_type": "speak", "target": null, "content": "shares a short update."}',
}

with httpx.Client(base_url='http://127.0.0.1:8000') as client:
    created = client.post('/simulations', json=payload)
    created.raise_for_status()
    simulation = created.json()
    print(json.dumps(simulation, indent=2))


In [ ]:
simulation_id = simulation['simulation_id']

with httpx.Client(base_url='http://127.0.0.1:8000') as client:
    while True:
        status_payload = client.get(f'/simulations/{simulation_id}').json()
        if status_payload['status'] == 'completed':
            break
        time.sleep(0.05)
    print(json.dumps(status_payload, indent=2))
    print(client.get(f'/simulations/{simulation_id}/agents').json())
    print(client.get(f'/simulations/{simulation_id}/agents/agent-0/memory').json())


## WebSocket Stream

Use any standard client against `ws://127.0.0.1:8000/simulations/{simulation_id}/stream`.

Example with `wscat`:

```bash
npx wscat -c ws://127.0.0.1:8000/simulations/$SIMULATION_ID/stream
```
